# Script unificado do pipeline
Esse notebook unifica todos os scripts do pipeline, para deixa-lo mais eficiente. Qualquer dúvida, é só rodar script por script seguindo o README, que da certo.

## Workflow
Para iniciar o pipeline, basta seguir os passos informados ao longo do notebook. Inicialmente, deve-se rodar os 2 próximos blocos de código, a fim de carregar os pacotes e funções usados ao longo do notebook.

In [ ]:
#--Carregando pacotes--#
import pandas as pd
import time
from tqdm import tqdm
from Bio import Entrez, SeqIO
import shutil, glob
import os
from rcsbapi.search import SeqSimilarityQuery
from rcsbapi.data import DataQuery

In [ ]:
#--Para carregar as funções--#

# Função para padronizar a tabela de proteínas de entrada
def fix_tabel(data):
    # Remove o id de acesso do pubchem CASO NECESSÁRIO
    #data.drop('Protein_Accession', axis=1, inplace=True)

    # Renomeia as colunas
    data = data.rename(columns= {'protid': 'protein', 'Taxonomy': 'organism',
                                 'Protein_Accession': 'pubchem_id'}) 

    # Removendo o conteúdo do último parenteses (nome do organismo), utilizando expressões regulares
    # r'' para abrir a regex
    # \s* captura o espaço antes do parênteses
    # \( pega o parenteses de abertura
    # [^)]* pega todo conteúdo que não seja ")"
    # *\) pega o fechamento do parenteses
    # \s* pega o espaço após o parenteses
    # $ garante que essa seleção ocorra somente na última ocorrência de parênteses 
    data["protein"]=data["protein"].str.replace(r'\s*\([^)]*\)\s*$', '', regex=True)

    # Na coluna de organismo, pega apenas as 2 primeiras palavras
    data["organism"]=data["organism"].str.split().str[:2].str.join(' ')
    return data

# Função de busca usando API do NCBI
def busca_ncbi(nome_proteina, organismo, id_pubchem):
    try:
        if not id_pubchem:
            # Função para encontrar os IDs no NCBI (caso não exista)
            handle_busca = Entrez.esearch(
                db="protein",
                term=f'"{nome_proteina}"[Protein Name] AND "{organismo}"[Organism]',
                retmax=1
            )
            resultado = Entrez.read(handle_busca) #Enterz abre a função e executa
            handle_busca.close() # Fecha a função

            # Pega apenas os ids encontrados, e coloca em uma variável
            ids_encontrados = resultado["IdList"]

            # Caso não encontre nenhum ID, retorna None
            if not ids_encontrados:
                return None
            
            id_usado = ids_encontrados[0] # Apenas o primeiro ID encotrado

        # No caso de ter ID do PubChem
        else:
            id_usado = id_pubchem
        
        # Agora, a partir dos IDs encontrados, realizar a busca pelo Fasta
        handle_fasta = Entrez.efetch(
            db = "protein",
            id=id_usado,
            rettype = "fasta",
            retmode="text"
        )
        fasta = handle_fasta.read().strip() # Limpa o fasta encontrado
        handle_fasta.close() # Como antes, tem que fechar a função

        time.sleep(0.4) # Respeita o tempo do API

        # Caso for um fasta válido (começa com ">"), o retorna
        return fasta if fasta.startswith(">") else None
    
    # Em caso de erro complexo, envolvendo a entrada e/ou API
    except Exception as e:
        if id_pubchem:
            print(f'\nErro NCBI [{nome_proteina} | {organismo} | {id_pubchem}]: {e}')
        else:
            print(f'\nErro NCBI [{nome_proteina} | {organismo}]: {e}')
        time.sleep(1)
        return None

# Função para integrar os mecanísmos de busca, Loopar as procuras, 
# escrever o .fasta com as sequências e relatório das buscas
def recuperar_sequencias(dados, arquivo_saida, comp):
    # Listas controle, para serem preenchidas com as sequências e relatório
    sequencias_encontradas = []
    relatorio = []

    # Loop usando o tqdm (para mostrar barra de progresso), para executar as buscas em cada proteína
    for _,linha in tqdm (dados.iterrows(), total=len(dados), desc="Buscando Proteínas"):
        
        # variáveis para armazenar nome da proteína e organismo
        nome = linha["protein"]
        org = linha["organism"]

        # variáveis de controle
        fasta = None # para a sequência
        fonte = None # para o relatório

        # Caso deseja realizar a busca pelo ID do Pubchem
        id_pc = linha["pubchem_id"]
        fasta = busca_ncbi(nome_proteina=nome, organismo=org, id_pubchem=id_pc)
        if fasta:
            fonte = "NCBI"

        # Caso encontre um fasta, coloca na lista de sequências, e adiciona ao relatório
        if fasta:
            sequencias_encontradas.append(fasta)
            relatorio.append({
                "Proteína":  nome,
                "Organismo": org,
                "PubChemID": id_pc,
                "Status":    "encontrada",
                "Fonte":     fonte
                })

        # Se não encontrar, coloca apenas no relatório
        else:
            relatorio.append({
                "Proteína":  nome,
                "Organismo": org,
                "PubChemID": id_pc,
                "Status":    "não encontrada",
                "Fonte":     None
                })
    
    # Após todas as buscas, escreve o .fasta
    with open(arquivo_saida, "w") as f:
        f.write("\n\n".join(sequencias_encontradas))
    
    # Para criar o dataframe do relatório
    df_relatorio = pd.DataFrame(relatorio)
    df_relatorio.to_csv(f"{comp}_relatorio_busca.csv", index=False)

    total = len(dados)
    encontradas = df_relatorio[df_relatorio["Status"] == "encontrada"].shape[0]
    
    # Printando um apanhado geral no terminal
    print(f'\n Concluído: {encontradas}/{total} proteínas recuperadas')
    print(f'\n-> NCBI: {df_relatorio[df_relatorio['Fonte']=='NCBI'].shape[0]}')
    print(f'\n-> Não Encontradas: {total - encontradas}')

    # A função retorna o relatório em dataframe, caso queira printar no terminal
    return df_relatorio

# Função para limpar os IDs
def limpar_id(id_sujo):
    # Inicialmente assumindo que não tem id unipro
    uniprot_id = None
    
    # Separar o id, caso esteja com muita info
    if "|" in id_sujo:
        partes = id_sujo.split("|")

        # Seleciona somente o ID por si só
        id_limpo = partes[1]

        # identificadores de ID uniprot
        if partes[0] in ("sp", "tr"):
            uniprot_id = partes[1]
    else:
        id_limpo = id_sujo # Caso o id ja esteja "limpo"
    
    return id_limpo, uniprot_id

# Função para extrair as sequências, a partir do ID (neste caso ID do Trypdb, presente no proteoma carregado)
def busca_id_proteoma(id, proteoma_usado):
    
    # Busca diretamente o id no proteoma
    if id in proteoma_usado:
        return proteoma_usado[id]

    # Se não encontrar o id, tenta encontrar parte dele (id do trypdb pode aparecer como sujo ou parte do orig)
    for chave in proteoma_usado:
        if id in chave or chave in id:
            return proteoma_usado[chave]

    return None # Caso não encontre de jeito nenhum


## Encontrando proteínas alvo de estudo sobre moleculas identidade ou similares
Inicialmente, deve-se procurar o smiles da molécula desejada no [PubChem](https://pubchem.ncbi.nlm.nih.gov), e buscar nas abas de "Identity" e "Similarity" pela opção de "Proteins" em "Linked Data". Essa opção mostra proteínas alvo dos estudos envolvendo a molécula identidade (se existir) ou similiares ao smiles forncecido.

## Extraindo as sequências das proteínas alvo referência
Em seguida, deve-se baixar o .csv das proteínas, contendo as informações de "protid", "Taxonomy" e "Protein_Accession". Após baixar os arquivos, deve-se preparar eles para o script no próximo bloco. Para preparar os arquivos:
- Dentro de "proteinas_triagem", criar um diretório com o nome identificador do composto (Ex.: RMS)
- Mover os arquivos baixados para a pasta criada
- Os arquivos devem seguir um identificador, se a origem foi de moléculas identidade ou similar (Ex.: RMS_I.csv)

A partir dai, é só executar o script, informando a lista de compostos com os nomes dos identificadores de cada csv de cada molécula testada.

O script a seguir então retornará um diretório contendo os fastas de cada proteína alvo referência encontrado para cada composto, além de um diretório contendo os relatórios da busca,

In [ ]:
#--Script para fasta das referências--#

# Input para entrar com o e-mail utilizado para validar o API do NCBI
Entrez.email = str(input("Entre com o email para o Enterz: "))

# Lista de compostos utilizados
entrada = input("Insira o nome dos compostos testados, separando-os por ',' no formato composto_I(S):")
lista_comps = [item.strip() for item in entrada.split(',')]

for i in range(len(lista_comps)):
    composto = lista_comps[i]

    print(f'\nExecutando buscas para o arquivo {composto}.csv')

    # Variável para aramazenar apenas o nome do composto
    nome_composto=composto.replace("_"," ").split()[0]

    # Abrindo a lista com o panda
    tabela = pd.read_csv(f"proteinas_triagem/{nome_composto}/{composto}.csv")

    # Arrumando o df
    tabela = fix_tabel(tabela)

    # Variável para armazenar o nome do arquivo fasta com as sequências
    arquivo = str(f"{composto}_proteinas.fasta")

    # Rodando a função principal
    relat_final=recuperar_sequencias(dados=tabela, arquivo_saida=arquivo, comp=composto)

    print(relat_final)

# Criando pastas e realocando os fastas e relatórios para as determinadas pastas
os.makedirs('relatorios_busca', exist_ok=True)
for arquivo in glob.glob('*relatorio_busca.csv'):
    shutil.move(arquivo, 'relatorios_busca/')

os.makedirs('fastas_ref', exist_ok=True)
for arquivo in glob.glob('*proteinas.fasta'):
    shutil.move(arquivo, 'fastas_ref/')


## BLAST com proteoma alvo
Após obter os fastas das referêncais, parte-se para o BLAST em proteoma do organismo alvo, com o objetivo de encontrar proteínas similares às referências no proteoma alvo.

**IMPORTANTE:** O blast é feito localmente em BASH. Recomenda-se criar um ambiente virtual nomeado "blast_env" utilizando mamba e miniconda3, e instalar neste ambiente o blastp (versão testada: 2.16.0+).

Para criar o ambiente: `mamba create -n blast_env python=3.12.3` (versão do python que utilizei)

Ative o ambiente com: `conda activate blast_env`

É necessário baixar o proteoma referência e realizar a indexação. Para a indexação, basta rodar o seguinte código:
```
#!/bin/bash

makeblastdb \
   -in #caminho para o fasta do proteoma
   -dbtype prot
   -out #pasta de output/nome do arquivo Ex: db_lla/Leishmania_amazonensis
   -title #"(título do proteoma)" Ex: "Leishmania amazonensis proteome"

```

Em seguida, basta rodar o script [blast_script.sh](blast_script.sh), que então todo o processo será realizado, com os outputs em "blast_results", no formato .tsv. Deve-se chamar o script e fornecer como primeiro argumento o caminho do proteoma indexado, da mesma forma que foi gerado na indexação, após a flag "-out". Como no exemplo acima, se a flag -out teve como argumento **db_lla/Leishmania_amazonensis**, o primeiro argumento do script será **db_lla/Leishmania_amazonensis**.


## Filtrando os resultados do BLAST
Com os resultados do blast em mãos, é necessário filtrar os melhores hits.

Os filtros que utilizei foram: Identidade >= 30%; Cobertura de alinhamento >= 70%; E-value <= 1.10^-5.

Rodando o seguinte script, os filtros são aplicados e os melhores hits ÚNICOS são extraidos, sendo aramzenados em arquivos .csv no diretório blast_filtrado.

In [ ]:
#--Filtrar melhores hits do Blast--#

# Variáveis para diretórios
results_dir = "blast_results/"
output_dir = "blast_filtrado/"
os.makedirs(output_dir, exist_ok=True)

# Variáveis filtro
ident_min = 30 # % identidade mínima
cover_min = 70 # % cobertura de alinhamento mínima
e_val_max = 1e-5 # e-value máximo

# Colunas dos tsv de resultados do blast
colunas = ["qseqid","sseqid","pident","qcovs","length","evalue","bitscore","stitle"]
# armazenando o nome dos arquivos de resultado blast
arquivos_tsv = sorted(glob.glob(f"{results_dir}*.tsv"))

# Loop para operar sobre os arquivos
for tsv in arquivos_tsv:
    # Extraindo apenas o nome do composto
    nome_base = os.path.basename(tsv).replace("_proteinas_blast_result.tsv", "")
    print(f'\nProcessando resultados do BLAST para {nome_base}.')

    df = pd.read_csv(tsv, sep="\t", names=colunas) # Ler o tsv

    print(f"    Hits brutos: {len(df)}")

    # Aplicando o filtro
    df_filtrado = df[
        (df["pident"] >= ident_min) &
        (df["qcovs"] >= cover_min) &
        (df["evalue"] <= e_val_max)
    ].copy()

    print(f"    Hits pós filtro: {len(df_filtrado)}")

    if df_filtrado.empty:
        print(" Nenhum hit passou no filtro.")
        continue

    # Selecionando apenas os melhores hits individuais de acordo com o bitscore 
    df_melhores = (
        df_filtrado.sort_values("bitscore", ascending=False)
        .drop_duplicates(subset="qseqid", keep="first")
        .reset_index(drop=True)
    )

    print(f"    Melhores hits únicos: {len(df_melhores)}")

    # Limpando os indices, e adicionando tais colunas
    df_melhores.insert(3, "id_limpo_hit", df_melhores["sseqid"].apply(lambda x: limpar_id(x)[0]))
    df_melhores.insert(4, "id_uniprot_hit", df_melhores["sseqid"].apply(lambda x: limpar_id(x)[1]))

    # Renomeando colunas
    df_melhores = df_melhores.rename(columns={
        "qseqid": "id_query",
        "sseqid": "id_bruto_hit",
        "pident": "identidade",
        "qcovs": "cobertura_query",
        "length": "tamanho_align",
        "stitle": "info_hit" 
    })

    # Reordenando colunas
    df_melhores = df_melhores[[
        "id_query",
        "id_bruto_hit",
        "id_limpo_hit",
        "id_uniprot_hit",
        "identidade",
        "cobertura_query",
        "tamanho_align",
        "evalue",
        "bitscore",
        "info_hit"
    ]]

    # Escrevendo arquivos de saída
    saida = f"{output_dir}{nome_base}_blast_filtrado.csv"
    df_melhores.to_csv(saida, index=False)
    print(f"    Resultados do BLAST filtrados salvos em: {saida}")

    del df


## Obtendo FASTAs dos *Hits*
Tendo as IDs dos hits, nos arquivos .csv, o próximo passo é rodar o script seguinte.

Este script realiza a busca dos IDs de cada hit dentro do proteoma, no formato FASTA, contido no diretório do proteoma alvo. O script retorna um arquivo .fasta para cada arquivo .csv filtrado do BLAST na pasta fastas_hits.

In [ ]:
#--Obter FASTAs dos Hits--#

# Variáveis diretório e caminho
input_dir = "blast_filtrado/"
path_proteoma = str(input("Insira o caminho para o fasta do proteoma alvo: "))
output_dir = "fastas_hits/"

os.makedirs(output_dir, exist_ok=True)

# Carregando o proteoma
print("Carregando o proteoma:")
proteoma = SeqIO.to_dict(SeqIO.parse(path_proteoma, "fasta"))
print(f"-> {len(proteoma)} sequêncais carregadas.")

# Extraindo os arquivos de input
arquivos_csv = sorted(glob.glob(f"{input_dir}*.csv"))

# Loop para executar as ações para cada arquivo input
for arquivo in arquivos_csv:
    
    # Extraindo o nome do arquivo
    nome_base = os.path.basename(arquivo).replace("_blast_filtrado.csv", "")
    print(f"\nProcessando {nome_base}.\n")

    # Abrir o arquivo como df
    df = pd.read_csv(arquivo)

    # Listas importantes
    sequencias_extraidas = []
    nao_encontradas = []

    # Loop para cada linha (proteína) em cada arquivo
    for _, row in df.iterrows():

        id_limpo = row["id_limpo_hit"] # Extraindo o ID

        sequencia = busca_id_proteoma(id_limpo, proteoma) # Buscando o FASTA relacionado ao ID

        if sequencia is None:
            print(f"    Não encontrado no proteoma: {id_limpo}")
            nao_encontradas.append(id_limpo)
            continue

        # Adicionando a sequência encontrada à lista
        sequencias_extraidas.append(sequencia)
        str_sequencia = str(sequencia.seq)

        print(f"    {id_limpo} encontrada! ({len(str_sequencia)} aa)")

    # Salvar FASTA com as sequencias extraidas
    fasta_saida = f"{output_dir}{nome_base}_sequencias_hits.fasta"
    SeqIO.write(sequencias_extraidas, fasta_saida, "fasta")
    print(f"\n    FASTA salvo em: {fasta_saida} ({len(sequencias_extraidas)} sequências).")

del df

## Obtendo FASTAs dos melhores *Hits*

Por fim, o script abaixo irá retornar (de acordo com um score próprio que leva em consideração identidade>cobertura>bitscore) as X melhores proteínas (a depender do usuário). Este script retorna os fastas com as melhores proteínas para cada composto. 

Além disso, ele realiza uma busca no API do PDB, procurando o melhor match para o FASTA de cada um dos top *Hits*. Isso é bom para ter um norte do nome da proteína, caso não se tenha informação sobre ou melhor, encontrar a proteína depositada no PDB destes melhores Hits, poupando a etapa de modelagem *in silico* de tal proteína. 

In [ ]:
#--Obter os FASTAs dos melhores Hits--#

input_dir = "blast_filtrado/"
output_dir = "fastas_melhores_hits/"

os.makedirs(output_dir, exist_ok=True)

# Quantas proteínas triar
top_melhores = int(input("Quantas proteínas quer retornar como melhores hits? "))

# Puxando os arquivos
arquivos = sorted(glob.glob(f"{input_dir}*.csv"))

for arquivo in arquivos:

    # Extraindo nome e abrindo o arquivo como df
    nome = os.path.basename(arquivo).replace("_blast_filtrado.csv", "")
    df = pd.read_csv(arquivo)

    # Estabelecendo o score para triagem
    # Valores dos pesos podem estar sujeito a mudanças, de acordo com a prioridade
    df["score_triagem"]=(
        df["identidade"]*0.5 + df["cobertura_query"]*0.3 + (df["bitscore"]/df["bitscore"].max())*100*0.2
    ) 

    # Criando df ordenado pelo score da triagem
    df_ordenado = df.sort_values("score_triagem", ascending=False)

    print(f"\n{nome} - Top {top_melhores}")
    print(df_ordenado[[
        "id_query", "id_limpo_hit", "identidade",
        "cobertura_query", "bitscore", "score_triagem"
    ]].head(top_melhores).to_string(index=False)) # Printa apenas os Top melhores 

    # Agora, para colocar as sequências das top melhores em um único fasta por composto
    sequencias = []

    # Iterando cada ID das top melhores
    for id_hit in df_ordenado["id_limpo_hit"].head(top_melhores):
        # Usando a funçãozinha
        sequence = busca_id_proteoma(id_hit, proteoma)
        sequencias.append(sequence)

    # Escrevendo o fasta
    fasta_saida = f"{output_dir}{nome}_melhores_hits.fasta"
    SeqIO.write(sequencias, fasta_saida, "fasta")
    print(f"\n    {top_melhores} Melhores FASTAS salvos em {fasta_saida}")

    # Procurando por proteínas catalogadas no PDB que são similares às melhores sequências
    # Isso é principalmente para ter uma estrutura para se basear para os próximos passos
    # E também pra dar nome às proteínas que selecionei
    ## Como trabalho com LLa muito provavelmente não vou encontrar proteínas dessa espécie catalogadas no PDB
    ### Mas similares já serve pra dar um norte
    sequencias_busca = []

    # Pega APENAS a sequência dentro do arquivo fasta, e transforma em STR
    # Será usado pelo API do PDB
    for seq_record in SeqIO.parse(fasta_saida, "fasta"):
        sequencias_busca.append(str(seq_record.seq))
    
    print(f"\nIDs PDB para proteínas similares às Top {top_melhores} para {nome}:")

    # Iterando para cada sequência
    for i in range(len(sequencias_busca)):

        # Estabelecendo o query para a busca no API
        # O identity cutoff é opcional e subjetivo de acordo com o objetivo do estudo
        query = SeqSimilarityQuery(
            value=sequencias_busca[i],
            identity_cutoff=0.3
        )

        # Como resultado, quero o polymer_entity (ID PDB)
        results = list(query(return_type="polymer_entity"))
        
        # Agora, se for encontrado, realiza busca de metadados (para extrair o nome da proteína)
        if len(results)>0:
            entrada = results[0][:4] # Limpao ID

            # Outro API de busca do PDB, agora para extrair metadados
            data_query = DataQuery(
                input_type="entries",
                input_ids=[entrada],
                return_data_list=["struct.title"] # Retorana elementos do título
            )

            # Executando o API de busca
            metadados = data_query.exec()
            
            entries = metadados["data"]["entries"]

            # Agora, printando no terminal o ID PDB e nome da proteína, se existir
            if entries and len(entries) > 0:    
                entry = entries[0]
                id_pdb = entry['rcsb_id']
                nome_proteina = entry['struct']['title']
                url_pdb = f"https://www.rcsb.org/structure/{id_pdb}"
                print(f"\n    {i+1}ª - {id_pdb} - {nome_proteina}\n        URL: {url_pdb}")
            
            else:
                print(f"\n    {i+1}ª - Não foi possível localizar esta proteína")
        
        time.sleep(0.4) # Respeitando o tempo do API


## Próximos passos

Este é o fim da triagem pelas proteínas alvo!

Agora, é seguir para a etapa da modelagem das proteínas que não foram encontradas no PDB, para então realizar o *docking*.